# Tech Department Agent - RAG + Memory Test

This notebook tests the Streamlit tech department agent implementation.

It verifies:

- Tech department policies are ingested into ChromaDB
- RAG retrieval returns relevant policy sections
- Agent commands work
- Device request tickets are created
- Device issue tickets are created and prioritized
- Interactions are stored as episodic memory
- Agent facts and operating rules are stored as declarative memory

The test uses a temporary database directory, so it does not modify the Streamlit app's real `.tech_agent_memory/` data.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from tech_agent import DEFAULT_KB_PATH, OllamaClient, TechDepartmentAgent, TechMemoryRAG

print(f"Policy file: {DEFAULT_KB_PATH}")
print(f"Policy file exists: {DEFAULT_KB_PATH.exists()}")
assert DEFAULT_KB_PATH.exists(), "Missing tech department policy file"

C:\Users\Saeed\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Policy file: E:\Projects\Vector_Database_Memory\knowledge_base\tech_department_policies.md
Policy file exists: True


## 1. Initialize Agent With Temporary Vector Memory

In [2]:
temp_dir = TemporaryDirectory(ignore_cleanup_errors=True)
test_db_dir = Path(temp_dir.name) / "tech_agent_test_db"

rag = TechMemoryRAG(db_dir=test_db_dir, kb_path=DEFAULT_KB_PATH)
llm = OllamaClient(model_name="mistral")
agent = TechDepartmentAgent(rag=rag, llm=llm)

stats = rag.stats()
print(stats)
assert stats["policies"] >= 5, "Expected policy sections to be ingested"
assert stats["episodic_memories"] == 0, "Temporary episodic memory should start empty"
assert stats["declarative_memories"] >= 1, "Agent should start with declarative operating memory"
assert stats["tickets"] == 0, "Temporary test ticket store should start empty"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9949.41it/s]

{'policies': 6, 'episodic_memories': 0, 'declarative_memories': 1, 'memories': 1, 'tickets': 0}


## 2. Test Policy RAG Retrieval

In [3]:
results = rag.search("How much is the laptop budget and how long does approval take?", include_memory=False)

for item in results[:3]:
    print(f"{item.kind} | {item.similarity:.2f} | {item.id}")
    print(item.content[:300], "\n")

combined = "\n".join(item.content.lower() for item in results)
assert "laptop" in combined, "RAG should retrieve laptop policy context"
assert "2000" in combined or "5 business days" in combined, "RAG should retrieve budget or timeline details"

policy | 0.64 | policy-0-device-request-policy
Device Request Policy
Employees can request laptops, desktops, monitors, keyboards, mice, headsets, docking stations, phones, and replacement chargers through the tech department.

All device requests must include the employee name, department, manager name, business reason, requested device type, p 

policy | 0.32 | policy-2-access-and-software-policy
Access and Software Policy
Software installation requests must include the tool name, vendor, business purpose, license need, data sensitivity, and manager approval.

Paid software requires budget owner approval before purchase.

Software that processes customer data, employee data, financial data,  

policy | 0.31 | policy-1-device-issue-and-support-policy
Device Issue and Support Policy
Employees should report device issues as soon as possible. The request must include the device type, asset tag if available, operating system, issue summary, error messages, urgency, and troubleshooting alr

## 3. Test Commands

In [4]:
help_result = agent.answer("/help")
stats_result = agent.answer("/stats")
policy_result = agent.answer("/policy remote work equipment")
remember_result = agent.answer("/remember Always ask for the asset tag before troubleshooting company laptops.")

print(help_result["answer"])
print(stats_result["answer"])
print(policy_result["answer"][:500])
print(remember_result["answer"])

assert "/policy" in help_result["answer"], "Help command should list policy command"
assert "policy sections" in stats_result["answer"], "Stats command should report policy count"
assert len(policy_result["sources"]) > 0, "Policy command should return retrieved sources"
assert "Declarative memory saved" in remember_result["answer"], "Remember command should save declarative memory"

Available commands:
- /policy <question> searches tech policies.
- /memory <question> searches policies, tickets, declarative memory, and episodic memory.
- /remember <fact or instruction> saves declarative memory for future answers.
- /stats shows memory counts.
- /request starts a device request.
- /issue starts a device issue report.


Memory status: 6 policy sections, 0 tickets, 1 episodic memories, 1 declarative memories.
Based on the tech department policy and memory I found:
- Remote Work Equipment Policy: Remote and hybrid employees may request one monitor, keyboard, mouse, headset, webcam, and docking station when their role requires regular remote work.
- Device Return Policy: Employees must return company devices, chargers, security keys, phones, and accessories on their last working day or by the date agreed with HR.
- Device Issue and Support Policy: Employees should report device issues as soon as possib
Declarative memory saved as declarative-9f4c390f-0fc0-4596-889b-1984b15ac4f6. I will use it as remembered tech department knowledge.


## 4. Test Device Request Ticket

In [5]:
request_result = agent.submit_device_request(
    {
        "employee": "Aisha Khan",
        "department": "Engineering",
        "manager": "Ravi Mehta",
        "device": "Laptop",
        "work_mode": "Hybrid",
        "business_reason": "Needs a development laptop for backend work",
        "preferred_date": "2026-09-20",
    }
)

print(request_result)
assert request_result["ticket_id"].startswith("TECH-"), "Device request should return a TECH ticket ID"
assert "Device request created" in request_result["answer"], "Device request answer should confirm creation"

{'ticket_id': 'TECH-20260912-7CC319', 'answer': 'Device request created as TECH-20260912-7CC319. Tech will review manager approval, business reason, budget fit, and target date before fulfillment.'}


## 5. Test Device Issue Ticket and Priority Classification

In [6]:
issue_result = agent.submit_device_issue(
    {
        "employee": "Aisha Khan",
        "device_asset": "LT-ENG-1042",
        "urgency": "High",
        "issue_summary": "VPN is blocked before a customer meeting",
        "troubleshooting": "Restarted laptop and VPN client",
    }
)

print(issue_result)
assert issue_result["ticket_id"].startswith("TECH-"), "Device issue should return a TECH ticket ID"
assert issue_result["priority"] == "high", "VPN or meeting issue should be classified as high priority"

{'ticket_id': 'TECH-20260912-091757', 'answer': 'Device issue logged as TECH-20260912-091757 with high priority. Include screenshots or exact error messages if available so support can triage faster.', 'priority': 'high'}


## 6. Test Memory and Ticket Retrieval

In [7]:
final_stats = rag.stats()
print(final_stats)

memory_results = rag.search("Aisha VPN laptop engineering", include_memory=True)
for item in memory_results[:5]:
    print(f"{item.kind} | {item.similarity:.2f} | {item.id}")
    print(item.content[:250], "\n")

assert final_stats["tickets"] == 2, "Expected one request ticket and one issue ticket"
assert final_stats["episodic_memories"] >= 5, "Commands and ticket actions should create episodic memory entries"
assert final_stats["declarative_memories"] >= 2, "Seeded and user-taught facts should create declarative memory"
assert any(item.kind == "ticket" for item in memory_results), "Memory search should retrieve created tickets"
assert any(item.kind == "declarative" for item in memory_results), "Memory search should retrieve declarative memory"

{'policies': 6, 'episodic_memories': 7, 'declarative_memories': 2, 'memories': 9, 'tickets': 2}
ticket | 0.55 | TECH-20260912-091757
Ticket TECH-20260912-091757
Type: device_issue
Employee: Aisha Khan
Device Asset: LT-ENG-1042
Urgency: High
Issue Summary: VPN is blocked before a customer meeting
Troubleshooting: Restarted laptop and VPN client 

ticket | 0.48 | TECH-20260912-7CC319
Ticket TECH-20260912-7CC319
Type: device_request
Employee: Aisha Khan
Department: Engineering
Manager: Ravi Mehta
Device: Laptop
Work Mode: Hybrid
Business Reason: Needs a development laptop for backend work
Preferred Date: 2026-09-20 

policy | 0.34 | policy-0-device-request-policy
Device Request Policy
Employees can request laptops, desktops, monitors, keyboards, mice, headsets, docking stations, phones, and replacement chargers through the tech department.

All device requests must include the employee name, department, manag 

policy | 0.30 | policy-3-remote-work-equipment-policy
Remote Work Equipment Po

## 7. Optional Ollama Answer Test

If Ollama with `mistral` is running, this cell checks a real generated answer. If Ollama is not available, the agent uses its policy-grounded fallback answer.

In [8]:
answer_result = agent.answer("Can I get a new laptop for hybrid work and what details do you need?")

print(answer_result["answer"])
print(f"Sources returned: {len(answer_result['sources'])}")

assert len(answer_result["answer"]) > 50, "Agent should produce a useful answer"
assert len(answer_result["sources"]) > 0, "Agent should retrieve policy or memory context"

temp_dir.cleanup()
print("All tech department agent tests passed.")

Based on the tech department policy and memory I found:
- Ticket TECH-20260912-7CC319: Type: device_request Employee: Aisha Khan Department: Engineering Manager: Ravi Mehta Device: Laptop Work Mode: Hybrid Business Reason: Needs a development laptop for backend work Preferred Date: 2026-09-20.
- Device Request Policy: Employees can request laptops, desktops, monitors, keyboards, mice, headsets, docking stations, phones, and replacement chargers through the tech department.
- User asked: /policy remote work equipment: Agent answered: Based on the tech department policy and memory I found: - Remote Work Equipment Policy: Remote and hybrid employees may request one monitor, keyboard, mouse, headset, webcam, and docking station when their role requires regu.
- Remote Work Equipment Policy: Remote and hybrid employees may request one monitor, keyboard, mouse, headset, webcam, and docking station when their role requires regular remote work.

Please provide any missing details such as employ